# ⚡ The Flux Capacitor
## What *is* a Large Language Model?

**Section 2** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

Everybody has now seen a chatbot type an answer back at them. Almost nobody has
been shown **what is actually happening** while it does that.

It is not looking anything up. It is not searching a database. There is no
sentence stored anywhere inside it waiting to be found.

It is doing one thing, very fast, over and over:

> ### 🎯 Guessing the next small piece of text.

That's it. That is the whole trick. Everything else in this repository — agents,
tools, retrieval, the mapping demo, all of it — is built on top of that one
behaviour. So it is worth three minutes to see it happen for real.

```mermaid
flowchart LR
    A["Your text"] --> B["Split into<br/>tokens"]
    B --> C["Score every<br/>possible next token"]
    C --> D["Pick one"]
    D --> E["Add it to<br/>the text"]
    E --> F{"Finished?"}
    F -- "no" --> C
    F -- "yes" --> G["Done"]
```

**The Flux Capacitor is what makes time travel possible — but on its own it is
just a glowing box.** This notebook is about the glowing box. Steering it comes
next, in Section 3.

---

### What you need

| | |
|---|---|
| ⏱️ | About 3 minutes for the live path |
| 💾 | ~1.6 GB the first time (the model downloads itself) |
| 🎮 | A GPU helps, but this one is small enough to run without one |
| 🔑 | No API key. Nothing leaves your machine. |

---

# 🚦 LIVE DEMO PATH

*Four cells. Run them in order.*

---

## Setup

We are deliberately using a **very small** model — 0.8 billion parameters, about
1.4 GB. Small enough to start in a few seconds, and small enough that you could
run it on the laptop in front of you.

It is not smart. That is fine. We are not here to be impressed by the answer,
we are here to watch the machinery.

In [1]:
from delorean import llm, viz

model, tokenizer = llm.load_local("tiny")

VOCAB = llm.vocab_size(model)
print(f"\nThis model chooses between {VOCAB:,} different tokens.")

Loading Qwen3.5 0.8B onto NVIDIA GeForce RTX 3080 ...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Ready. Using 1.40 GB of 10.0 GB (fp16).

This model chooses between 248,320 different tokens.


---

## Act 1 — 🔢 Words become numbers

Before the model sees your question, your question stops being words.

It gets chopped into **tokens** — pieces roughly the size of a syllable — and
each piece is swapped for a number. The model only ever sees the numbers.

Watch what happens to a word every one of us uses every day.

In [2]:
viz.show_tokens(tokenizer, "appraisal")

Two pieces. `appr` and `aisal`. The model has no idea those belong together as a
word — it only knows that token 20,000-something is very often followed by token
40,000-something.

Now something from a real parcel record. 👀

In [3]:
viz.show_tokens(tokenizer, "The homestead exemption")
viz.show_tokens(tokenizer, "parcel 04-127-3891")

> ### ⚠️ Great Scott — look at that parcel number.
>
> **Thirteen tokens.** Every single digit is its own separate piece. The model
> does not see `04-127-3891` as an identifier. It sees thirteen unrelated
> fragments in a row.
>
> **Remember this.** It is the reason language models fumble parcel numbers,
> account numbers, and arithmetic — and we will come back to it in Section 5
> when we look at where these things break.

> ## 💡 The model has never seen a word. It sees numbers.

---

## Act 2 — 🎲 One prediction, 248,320 options

Here is the part nobody shows you.

Give the model some text and it produces a **score for every single token it
knows** — all 248,320 of them — saying how likely each one is to come next.

That is a real forward pass through the model. Three lines.

In [4]:
import torch


def next_token_odds(text, k=5):
    """Ask the model: what comes next? Return the top k candidates."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        scores = model(**inputs).logits[0, -1, :]   # a score for EVERY token

    odds = torch.softmax(scores.float(), dim=-1)    # turn scores into percentages
    top_odds, top_ids = torch.topk(odds, k)         # keep the best few

    return [(tokenizer.decode([i]), p)
            for p, i in zip(top_odds.tolist(), top_ids.tolist())]


def show(text, k=5):
    viz.show_probabilities(next_token_odds(text, k), prompt=text, vocab_size=VOCAB)

Let's start with one everybody in this room already knows the answer to. 🏘️

In [5]:
show("Location, location,")

·location,,81.1%
·and,,4.7%
·Location,,0.8%
·the,,0.7%
·I,,0.6%


So does the model. Not because it was told, and not because anyone stored that
phrase in it — but because in everything it ever read, that word followed those
words often enough to be the obvious bet.

Now something from our own profession.

In [6]:
show("An appraiser estimates the fair market")

·value,,99.4%
·price,,0.2%
·sale,,0.0%
·values,,0.0%
·rental,,0.0%


Practically a certainty. But notice it is **not** 100%. It is still a bet.

Now watch what happens when the answer is genuinely debatable. 🤔

In [7]:
show("The homestead exemption reduces a homeowner's property")

·tax,,52.7%
·value,,24.9%
·taxes,,10.4%
·costs,,1.1%
‑,,1.0%


Look at that spread. `tax`, `value`, `taxes` — and honestly, **all three would
be defensible** depending on your state.

The model is not confused. It is doing exactly what it always does: weighing
the evidence and reporting how strongly it leans. When it is sure, one bar
dominates. When it isn't, you can see the doubt.

> ## 💡 That's the whole trick. It looked at 248,320 options and picked one.

---

## Act 3 — 🔁 Do it again. And again. And again.

If the model only predicts *one* token, how do you get a paragraph?

You take the winner, glue it onto the end of the text, and **ask the same
question again**. Then again. Then again, until it decides to stop.

That loop is called *generation*. Below, each shaded block is one token
arriving. Watch the boundaries — those are the same pieces from Act 1.

In [8]:
from threading import Thread
from transformers import TextIteratorStreamer

question = [{"role": "user", "content": "In one sentence, what does a property appraiser do?"}]
prompt = llm.chat_prompt(tokenizer, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# The streamer hands us each token the moment it is produced, instead of making
# us wait for the whole answer.
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
canvas = viz.TokenCanvas(title="one token at a time")

# generate() blocks, so we run it on a background thread and read the stream here.
Thread(target=model.generate, kwargs=dict(
    **inputs, streamer=streamer, max_new_tokens=60, do_sample=False,
)).start()

for piece in streamer:
    canvas.add(piece)

> ## 💡 Everything you have ever seen ChatGPT do is this loop, running fast.

No magic. No understanding. A very good next-piece guesser, run a few hundred
times a second.

---

## Act 4 — 🏘️ Now ask it something real

Same loop, longer answer, boundaries turned off so it reads normally — the way
you are used to seeing it.

In [9]:
question = [{"role": "user", "content":
             "What are the three approaches to value in real property appraisal?"}]
prompt = llm.chat_prompt(tokenizer, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
canvas = viz.TokenCanvas(title="Qwen3.5 0.8B, running on this machine", boundaries=False)

Thread(target=model.generate, kwargs=dict(
    **inputs, streamer=streamer, max_new_tokens=220, do_sample=False,
)).start()

for piece in streamer:
    canvas.add(piece)

---

## 🎬 That's Section 2

A language model is a **next-token guesser**. A very sophisticated one, trained
on an enormous amount of text, but a guesser.

Two things to carry forward:

1. 🔢 **It sees tokens, not words** — which is why identifiers and arithmetic are
   a weak spot *(Section 5)*
2. 🎲 **Every answer is a chain of bets** — which is why the same question can
   give different answers *(and why "confident" does not mean "correct")*

And notice what it *didn't* do. It didn't know who it was. It didn't know it was
talking to an appraiser. It had no instructions at all.

> ### ⏭️ **Next: Section 4 — Setting the Time Circuits.**
> Punch a destination into the dashboard and the same box behaves completely
> differently.

---

---

# 🔬 EXPLORE ON YOUR OWN

*Everything below here is off the live path — for when you take this back to the
office. Nothing here is needed for the talk.*

---

## 5. 🧮 Why it fumbles parcel numbers and math

Act 1 showed that `04-127-3891` is thirteen separate tokens. Here is what that
costs you in practice.

In [10]:
viz.show_tokens(tokenizer, "$412,500")
viz.show_tokens(tokenizer, "1,247 + 3,891")

Every digit is its own token, and the comma is another. The model is not doing
arithmetic. It is guessing which digit usually follows which.

Sometimes that guess is right — and **that is what makes this dangerous.**
Let's give it three problems and check its work.

In [11]:
def ask_math(question, truth):
    """Ask the model to compute something, then check it against Python."""
    msgs = [{"role": "user", "content": f"What is {question}? Answer with just the number."}]
    inputs = tokenizer(llm.chat_prompt(tokenizer, msgs), return_tensors="pt").to(model.device)

    out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
    said = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:],
                            skip_special_tokens=True).strip().replace("\n", " ")

    correct = str(int(truth)) in said.replace(",", "").replace("$", "")
    mark = "CORRECT" if correct else "WRONG  "
    print(f"{mark} | {question:24} | model: {said[:40]:42} | truth: {truth:,.0f}")


ask_math("1,247 + 3,891",   1247 + 3891)
ask_math("1,247 x 3,891",   1247 * 3891)
ask_math("17% of $412,500", 0.17 * 412500)

CORRECT | 1,247 + 3,891            | model: 1,247 + 3,891 = 5,138                      | truth: 5,138


WRONG   | 1,247 x 3,891            | model: 1,247 x 3,891 = 4,860,057                  | truth: 4,852,077


WRONG   | 17% of $412,500          | model: 17% of $412,500 is 68,125.                 | truth: 70,125


The addition it gets right. It has seen sums like that a great many times.

Now look at the multiplication. 😬

It did not say *"I don't know."* It produced a confident, seven-digit,
correctly-shaped number — that is **wrong by several thousand dollars**. It
passes the eyeball test completely.

> ### ⚠️ Great Scott — that is the failure mode that should worry you.
>
> A model that fails loudly is easy to catch. A model that hands you
> `4,860,057` when the answer is `4,852,077` is not. **It doesn't look like an
> error. It looks like an answer.**
>
> Now imagine that number on a notice of value. 📬

> 🛠️ **The fix is not a smarter model — it is a calculator.** In Section 7 we
> hand the model a real Python function and let it do the arithmetic properly.
> That is what a *tool* is, and it is why the rest of this repository exists.

---

## 6. 🎰 Same question, different answers

In Act 2 the model produced a ranked list of candidates. So far we have always
taken the top one — that's called **greedy** decoding, and it is why the demo
gives the same answer every time.

But you don't have to. You can *sample* from the distribution instead: usually
pick the favourite, but sometimes take the runner-up. That is what "temperature"
controls, and it is why a chatbot feels less robotic.

Run this a few times. 🎲

In [12]:
question = [{"role": "user", "content": "Describe a house in one short sentence."}]
prompt = llm.chat_prompt(tokenizer, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("GREEDY — always take the top candidate (identical every run):\n")
for run in range(2):
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
    print(" ", tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip())

print("\nSAMPLING at temperature 1.0 — roll the dice each time:\n")
for run in range(3):
    out = model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=1.0, top_p=0.95)
    print(" ", tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip())

GREEDY — always take the top candidate (identical every run):



  A cozy, sun-drenched home nestled in a quiet valley, where the air smells of fresh pine and warm woodsmoke.


  A cozy, sun-drenched home nestled in a quiet valley, where the air smells of fresh pine and warm woodsmoke.

SAMPLING at temperature 1.0 — roll the dice each time:



  A cozy home nestled in a quiet forest corner, nestled in the forest,


  There is no "house" and no "home" in one sentence. However, a common sentence using a word like "furniture" is:


  The house sits quietly atop a mountain peak overlooking a valley where people gather to watch the sun set.


> 💡 **This matters for your office.** If you are producing a number that goes on
> a notice of value, you want the boring, repeatable setting. Creativity is a
> feature for writing; it is a liability for valuation.

---

## 7. ✏️ Try your own

Change the text below to anything you like and run it. Half-finished sentences
work best — you are asking "what comes next?"

Some to try:
- `"The subject property is located in"`
- `"Comparable sales should be adjusted for"`
- `"An appeal must be filed within"`

In [13]:
show("The subject property is located in")

·the,,36.6%
·a,,21.4%
·an,,2.1%
·**,,1.8%
:,,1.0%


---

## 8. 🏎️ Try a real model

Everything above ran on the little 0.8B model. Swap one word and you get the 4B
workhorse that powers the agent demos in Sections 7 through 11.

It is a bigger download (~8 GB) and needs about 3 GB of VRAM, but the code does
not change at all.

In [14]:
# Uncomment to run. First time downloads roughly 8 GB.
#
# big_model, big_tok = llm.load_local("qwen")     # <- the only change
#
# question = [{"role": "user", "content":
#              "What are the three approaches to value in real property appraisal?"}]
# prompt = llm.chat_prompt(big_tok, question)
# inputs = big_tok(prompt, return_tensors="pt").to(big_model.device)
#
# streamer = TextIteratorStreamer(big_tok, skip_prompt=True, skip_special_tokens=True)
# canvas = viz.TokenCanvas(title="Qwen3.5 4B", boundaries=False)
# Thread(target=big_model.generate, kwargs=dict(
#     **inputs, streamer=streamer, max_new_tokens=300, do_sample=False)).start()
# for piece in streamer:
#     canvas.add(piece)

---

## 9. 📋 What else could you run?

Every model in this repository's registry, with what it is good and bad at.
Use this to pick one for your own office.

In [15]:
from delorean import models

models.compare()


Qwen3.5 0.8B  (0.8B, fp16)
  Qwen/Qwen3.5-0.8B
  License : Apache 2.0
  VRAM    : 1.44 GB measured   Speed: ~14 tokens/sec
  Good at :
      + Starting up in about three seconds
      + Running on a laptop with no GPU at all
      + Showing how generation works, one token at a time
  Watch   :
      - Not for agent work — it is here to be understood, not to reason
      - Will confidently get facts wrong

Qwen3.5 4B  (4B, 4-bit NF4)
  Qwen/Qwen3.5-4B
  License : Apache 2.0
  VRAM    : 2.98 GB measured   Speed: ~14 tokens/sec
  Good at :
      + Tool calling — ranked 1st of 13 local models in a 2026 evaluation
      + Long documents (262,144 token context)
      + Punching far above its size; it beat models 5-7x larger
  Watch   :
      - Eager to call several tools at once when one at a time would do
      - Thinking mode is ON by default and adds a pause before answering

SmolLM3 3B  (3B, 4-bit NF4)
  HuggingFaceTB/SmolLM3-3B
  License : Apache 2.0
  VRAM    : not measured here
  Goo

---

<div align="center">

### 🕰️ *"Your future hasn't been written yet. No one's has.* 🕰️
### *Your future is whatever you make it. So make it a good one."*

🚗💨 ⚡ 🏘️

</div>